# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/imnxr/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task(type)


*Classification, clustering, ranking, or scoring — which one, and why?*



The primary task is **binary classification**.

For every content page in the dataset, the model predicts one of two classes:

- `1` — the page is declining and may need investigation or a content refresh.
- `0` — the page is not currently declining.

Although the formal task is classification, the model can also output a probability such as `0.82`. That probability can be used as a **priority score** to rank pages for review.

This framing is useful because a content team usually cannot manually investigate every page. The model narrows the list to pages that deserve attention first.


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

The proposed target column is:

is_declining_label
It is a binary proxy created from the observed trend direction:

is_declining_label = 1 when trend_direction == "down"
is_declining_label = 0 otherwise
This is a proxy target, not a perfect definition of business value. A downward trend does not automatically mean that a page should be rewritten. Seasonality, tracking changes, ranking volatility, or changes in search demand may also cause decline.

The model therefore supports investigation; it does not automatically decide that content must be changed.

## 3. Success metric

*One metric you can defend. What number means 'good'?*


The primary success metric is **F1 score for the declining class**.

F1 balances:

- **Precision:** Of the pages flagged as declining, how many were actually declining?
- **Recall:** Of all truly declining pages, how many did the model find?

A model with low precision wastes the content team's time through unnecessary alerts. A model with low recall misses pages that may require investigation.

For the first version, I would consider an **F1 score of at least 0.70 on held-out data** a useful result, provided it also performs better than a transparent baseline rule.

I would also inspect the confusion matrix and class distribution. Accuracy alone may be misleading if declining pages are uncommon.

## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*


The unit of analysis is one content-page observation per row.

Each row represents an anonymized content item or URL and contains observable signals that may include impressions, clicks, CTR, average position, engagement, content age, word count, or trend information.

The exact available columns are inspected below rather than assumed.

In [22]:
from pathlib import Path
import pandas as pd
import numpy as np

repo_root = Path.cwd()

candidate_paths = [
    repo_root / "data" / "raw" / "content_refresh_anonymized.csv",
    repo_root.parent / "data" / "raw" / "content_refresh_anonymized.csv",
    repo_root.parents[1] / "data" / "raw" / "content_refresh_anonymized.csv"
    if len(repo_root.parents) > 1 else Path("__missing__"),
]

data_path = next((path for path in candidate_paths if path.exists()), None)

if data_path is None:
    raise FileNotFoundError(
        "Could not find content_refresh_anonymized.csv. "
        "Confirm the actual dataset path in the starter repository."
    )

df = pd.read_csv(data_path)

print(f"Loaded: {data_path}")
print(f"Shape: {df.shape[0]:,} rows × {df.shape[1]:,} columns")
print("\nColumns:")
print(df.columns.tolist())

df.head()

Loaded: d:\Internship\flyrank-ml-internship\data\raw\content_refresh_anonymized.csv
Shape: 30,000 rows × 44 columns

Columns:
['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


In [23]:
preferred_columns = [
    "client_hash_id",
    "content_hash_id",
    "url_hash_id",
    "keyword_hash_id",
    "impressions",
    "clicks",
    "ctr",
    "position",
    "average_position",
    "sessions",
    "scroll_rate",
    "engagement_rate",
    "word_count",
    "content_age_days",
    "age",
    "trend_direction",
    "is_declining_label",
]

available_columns = [
    column for column in preferred_columns
    if column in df.columns
]

if not available_columns:
    available_columns = df.columns[:10].tolist()

unit_of_analysis_df = df[available_columns].head(10).copy()

print("Unit of analysis: one row represents one content-page observation.")
unit_of_analysis_df

Unit of analysis: one row represents one content-page observation.


,ctr,scroll_rate,engagement_rate,word_count,content_age_days,trend_direction
0,0.76,4.55,5.88,3221.0,187,down
1,0.05,10.00,0.00,2481.0,445,down
2,0.09,28.57,0.00,3515.0,141,down
3,0.49,3.45,1.28,NaN,463,stable
4,0.13,24.29,0.00,2803.0,263,down
5,0.03,25.00,0.00,3080.0,147,down
6,0.00,0.00,0.00,3059.0,90,down
7,0.06,7.14,3.57,NaN,445,stable
8,0.09,6.25,5.88,3807.0,90,down
9,0.16,0.00,0.00,NaN,257,down


### Interpretation of one row

One row represents one content item/page observation in the starter dataset. The feature columns describe the page's observed search or content behaviour, while the target indicates whether its trend is declining.

The eventual model would learn patterns across many rows and estimate the probability that a new or held-out page belongs to the declining class.


In [24]:
# Create or validate the proposed binary target.

if "is_declining_label" in df.columns:
    framed_df = df.copy()
    print("Using the existing is_declining_label column.")
elif "trend_direction" in df.columns:
    framed_df = df.copy()
    framed_df["is_declining_label"] = (
        framed_df["trend_direction"]
        .astype(str)
        .str.strip()
        .str.lower()
        .eq("down")
        .astype(int)
    )
    print("Created is_declining_label from trend_direction.")
else:
    framed_df = df.copy()
    framed_df["is_declining_label"] = pd.Series(
        [pd.NA] * len(framed_df), dtype="Int64"
    )
    print(
        "No trend_direction column was found. "
        "A placeholder target column was created for framing only."
    )

target_preview_columns = [
    c for c in [
        "content_hash_id",
        "url_hash_id",
        "impressions",
        "clicks",
        "ctr",
        "position",
        "average_position",
        "trend_direction",
        "is_declining_label",
    ]
    if c in framed_df.columns
]

framed_df[target_preview_columns].head(10)


Created is_declining_label from trend_direction.


,ctr,trend_direction,is_declining_label
0,0.76,down,1
1,0.05,down,1
2,0.09,down,1
3,0.49,stable,0
4,0.13,down,1
5,0.03,down,1
6,0.00,down,1
7,0.06,stable,0
8,0.09,down,1
9,0.16,down,1


### Leakage note

`trend_direction` is used only to define the proxy target. It must not be included as a model input feature, because doing so would reveal the answer to the model and cause target leakage.

In [25]:
# Inspect the target distribution.

target_counts = (
    framed_df["is_declining_label"]
    .value_counts(dropna=False)
    .rename_axis("is_declining_label")
    .reset_index(name="row_count")
)

target_counts["percentage"] = (
    target_counts["row_count"] / len(framed_df) * 100
).round(2)

target_counts


,is_declining_label,row_count,percentage
0,1,16262,54.21
1,0,13738,45.79


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

A fixed rule might say:

```text
Flag a page when clicks fall below a chosen threshold.
```

That rule is simple and transparent, but it has important weaknesses:

1. A low number of clicks may be normal for a niche page.
2. A page may still receive many clicks while declining sharply relative to its own history.
3. Multiple signals can interact. For example, impressions may rise while CTR falls, or ranking position may worsen while engagement stays strong.
4. The best threshold may differ across clients, topics, content ages, and search-demand levels.
5. A fixed rule does not naturally learn nonlinear combinations of signals from historical examples.

Machine learning is useful because it can learn patterns from several observable variables at the same time and estimate a probability of decline.

However, ML should only be retained if it performs better than a transparent baseline rule on held-out data. If a simple rule performs equally well, the simpler rule would be preferable because it is easier to explain, maintain, and audit.


## Content action supported by the output

The output supports a **content refresh prioritization action**.

A content strategist could:

1. Sort pages by predicted probability of decline.
2. Review the highest-priority pages.
3. Check the page's query, ranking, CTR, engagement, and content-age signals.
4. Decide whether to refresh, consolidate, internally link, re-optimize, monitor, or leave the page unchanged.

The model does not directly publish or rewrite content. It helps decide **where human investigation should begin**.


## Self-check

Before you submit, confirm each line honestly:

- [X] Every section above is filled — markdown thinking AND the code that backs it
- [X] The notebook runs top to bottom with no errors (Runtime → Run all)
- [X] No client names, URLs, or private queries anywhere
- [X] My claims use careful words: observed, measured, directional, decision-support
- [X] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

## Final framing statement

This lane is framed as a supervised binary-classification problem. Using page-level search and content signals, the model classifies whether a content-page observation exhibits a declining trend. Its positive-class probability can then prioritize pages for human investigation. This is currently a decision-support classification task rather than a claim about future decline.
